<a href="https://colab.research.google.com/github/bira69/TraderBot/blob/main/btc_price_predictor.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from transformers import pipeline

classifier = pipeline("sentiment-analysis")

result = classifier("I am actually breaking out of the gray zone today!")

print(result)

[transformers] No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f.
Using a pipeline without specifying a model name and revision in production is not recommended.


config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

[{'label': 'POSITIVE', 'score': 0.9850028157234192}]


In [ ]:
import yfinance as yf

btc_data = yf.download('BTC-USD', period='1y', interval='1d')

print(btc_data.head())

/tmp/ipykernel_675/4202311377.py:3: FutureWarning: YF.download() has changed argument auto_adjust default to True
  btc_data = yf.download('BTC-USD', period='1y', interval='1d')
[*********************100%***********************]  1 of 1 completed

Price               Close           High            Low           Open  \
Ticker            BTC-USD        BTC-USD        BTC-USD        BTC-USD   
Date                                                                     
2025-10-22  107688.585938  109115.132812  106778.000000  108491.531250   
2025-10-23  110069.726562  111288.593750  107548.429688  107679.437500   
2025-10-24  111033.921875  111842.531250  109770.148438  110069.351562   
2025-10-25  111641.726562  111947.703125  110704.406250  111032.617188   
2025-10-26  114472.445312  115260.906250  111268.484375  111639.054688   

Price            Volume  
Ticker          BTC-USD  
Date                     
2025-10-22  80807013218  
2025-10-23  54944076060  
2025-10-24  48160816980  
2025-10-25  24707667305  
2025-10-26  41708524143  


In [ ]:
# We can extract the 'Close' series specifically for 'BTC-USD' to avoid multi-index alignment issues
close_prices = btc_data['Close']['BTC-USD']

btc_data[('Tomorrow', 'BTC-USD')] = close_prices.shift(-1)
btc_data[('Target', 'BTC-USD')] = (btc_data[('Tomorrow', 'BTC-USD')] > close_prices).astype(int)


print(btc_data[[('Close', 'BTC-USD'), ('Tomorrow', 'BTC-USD'), ('Target', 'BTC-USD')]].head())

Price               Close       Tomorrow  Target
Ticker            BTC-USD        BTC-USD BTC-USD
Date                                            
2025-10-22  107688.585938  110069.726562       1
2025-10-23  110069.726562  111033.921875       1
2025-10-24  111033.921875  111641.726562       1
2025-10-25  111641.726562  114472.445312       1
2025-10-26  114472.445312  114119.328125       0


In [ ]:
from sklearn.ensemble import RandomForestClassifier

df = btc_data.dropna()

features = ['Open', 'High', 'Low', 'Close', 'Volume']

x = df[features]
y = df['Target']

model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(x, y)

accuracy = model.score(x, y)
print(f"Model Training Accuracy: {accuracy:.2%}")


/usr/local/lib/python3.13/dist-packages/sklearn/base.py:1389: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)


Model Training Accuracy: 100.00%


In [ ]:
train_size = int(len(df) * 0.8)
x_train, x_test = x.iloc[:train_size], x.iloc[train_size:]
y_train, y_test = y.iloc[:train_size], y.iloc[train_size:]

model.fit(x_train, y_train)

test_accuracy = model.score(x_test, y_test)
print(f"Realistic Test Accuracy: {test_accuracy:.2%}")


/usr/local/lib/python3.13/dist-packages/sklearn/base.py:1389: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)


Realistic Test Accuracy: 47.95%


In [ ]:
#Create df from the btc_data
df = btc_data.copy()

#Calculating 10 day moving average
df['SMA_10'] = df['Close'].rolling(window=10).mean()

#Calculating 10 day percentage return
df['Pct_change'] = df['Close'].pct_change()

#Drop raw with missning values created by the moving window
df = df.dropna()

#Update features list to include our new indicators
features = ['Open', 'High', 'Low', 'Close', 'Volume', 'SMA_10', 'Pct_change']

x = df[features]
y = df['Target']

#Split chronlogically (80% train, 20% test)
train_size = int(len(df) * 0.8)
x_train, x_test = x.iloc[:train_size], x.iloc[train_size:]
y_train, y_test = y.iloc[:train_size], y.iloc[train_size:]

#Train model and evaluate on unseen test data
model.fit(x_train, y_train)
test_accuracy = model.score(x_test, y_test)
print(f"Updated Test Accuracy: {test_accuracy:.2%}")

/usr/local/lib/python3.13/dist-packages/sklearn/base.py:1389: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)


Updated Test Accuracy: 49.30%


In [ ]:
import pandas as pd
import yfinance as yf
from  sklearn.ensemble import RandomForestClassifier

#Download 1 year of daily bitcoin data
btc_data = yf.download('BTC-USD', period='1y', interval='1d')

#Flattening multi-level columns if present
if isinstance(btc_data.columns, pd.MultiIndex):
  btc_data.columns = btc_data.columns.get_level_values(0)

#Target label (1 if tomorrow > today, else 0)
btc_data['Tomorrow'] = btc_data['Close'].shift(-1)
btc_data['Target'] = (btc_data['Tomorrow'] > btc_data['Close']).astype(int)

#Feature engineering: Moving averages and returns
df = btc_data.copy()

df['SMA_10'] = df['Close'].rolling(window=10).mean()
df['SMA_50'] = df['Close'].rolling(window=50).mean() #Long term trend
df['Pct_change'] = df['Close'].pct_change()

#14-day Relative strength index (RSI) calculation
delta = df['Close'].diff()
gain = delta.clip(lower=0).rolling(window=14).mean()
loss = (-delta.clip(upper=0)).rolling(window=14).mean()
rs = gain / loss
df['RSI'] = 100 - (100 / (1 + rs))

#Remove missing values created by 50-day rolling window and shift
df = df.dropna()

#Define expanded features set
features = ['Open', 'High', 'Low', 'Close', 'Volume', 'SMA_10', 'SMA_50', 'Pct_change', 'RSI']

x = df[features]
y = df['Target']

#Chronological Train/Test Split(80/20)
train_size = int(len(df) * 0.8)
x_train, x_test = x.iloc[:train_size], x.iloc[train_size:]
y_train, y_test = y.iloc[:train_size], y.iloc[train_size:]

#Train model and evaluate on unseen test data
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(x_train, y_train)
test_accuracy = model.score(x_test, y_test)
print(f"Updated Test Accuracy: {test_accuracy:.2%}")

/tmp/ipykernel_675/3655896102.py:6: FutureWarning: YF.download() has changed argument auto_adjust default to True
  btc_data = yf.download('BTC-USD', period='1y', interval='1d')
[*********************100%***********************]  1 of 1 completed


Updated Test Accuracy: 55.56%


In [ ]:
import pandas as pd

#Get feature importance scores from the fitted model
importances = model.feature_importances_

#Map scores to feature names and sort them descending
importance_df = pd.DataFrame({
    'Feature':features,
    'Importance':importances
}).sort_values('Importance', ascending=False)

#Print the sorted table
print(importance_df)


      Feature  Importance
7  Pct_change    0.135242
6      SMA_50    0.123876
8         RSI    0.121555
5      SMA_10    0.110514
4      Volume    0.110427
0        Open    0.103869
1        High    0.102146
2         Low    0.098029
3       Close    0.094342


In [ ]:
import pandas as pd
import yfinance as yf
from sklearn.ensemble import RandomForestClassifier

#Download data and flatten headers
btc_data = yf.download('BTC-USD', period='1y', interval='1d')
if isinstance(btc_data.columns, pd.MultiIndex):
  btc_data.columns = btc_data.columns.get_level_values(0)

#Create target label
btc_data['Tomorrow'] = btc_data['Close'].shift(-1)
btc_data['Target'] = (btc_data['Tomorrow'] > btc_data['Close']).astype(int)

#Feature engineering
df = btc_data.copy()
df['SMA_10'] = df['Close'].rolling(window=10).mean()
df['SMA_50'] = df['Close'].rolling(window=50).mean()
df['Pct_change'] = df['Close'].pct_change()

delta = df['Close'].diff()
gain = delta.clip(lower=0).rolling(window=14).mean()
loss = (-delta.clip(upper=0)).rolling(window=14).mean()
rs = gain / loss
df['RSI'] = 100 - (100 / (1 + rs))

df = df.dropna()

# Train / Test split
features = ['Open', 'High', 'Low', 'Close', 'Volume', 'SMA_10', 'SMA_50', 'Pct_change', 'RSI']
x = df[features]
y = df['Target']

train_size = int(len(df) * 0.8)
x_train, x_test = x.iloc[:train_size], x.iloc[train_size:]
y_train, y_test = y.iloc[:train_size], y.iloc[train_size:]

#Fit Random Forest
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(x_train, y_train)

#Run Trading Backtest
test_predictions = model.predict(x_test)
backtest_df = pd.DataFrame(index=x_test.index)
backtest_df['Prediction'] = test_predictions
backtest_df['Actual_Return'] = df.loc[x_test.index, 'Pct_change'].shift(-1)
backtest_df['Strategy_Return'] = backtest_df['Prediction'] * backtest_df['Actual_Return']

backtest_df = backtest_df.dropna()
buy_hold_total = (1 + backtest_df['Actual_Return']).cumprod().iloc[-1]
strategy_total = (1 + backtest_df['Strategy_Return']).cumprod().iloc[-1]

print(f"Buy and Hold Growth Factor: {buy_hold_total:.2f}x")
print(f"AI Strategy Growth Factor: {strategy_total:.2f}x")

/tmp/ipykernel_675/3186275267.py:6: FutureWarning: YF.download() has changed argument auto_adjust default to True
  btc_data = yf.download('BTC-USD', period='1y', interval='1d')
[*********************100%***********************]  1 of 1 completed


Buy and Hold Growth Factor: 1.20x
AI Strategy Growth Factor: 1.10x


In [ ]:
#Convert 0/1 predoctions inot -1/+1 trading positions
#(1 = Long / Buy, -1 = Short / Sell)
backtest_df['Position'] = backtest_df['Prediction'].map({1: 1, 0: -1})

#Calculate returns (Position * Actual Return)
backtest_df['Actual_Return'] = df.loc[x_test.index, 'Pct_change'].shift(-1)
backtest_df['Strategy_Return'] = backtest_df['Position'] * backtest_df['Actual_Return']

#Calculate total growth factor
backtest_df = backtest_df.dropna()
buy_hold_total = (1 + backtest_df['Actual_Return']).cumprod().iloc[-1]
strategy_total = (1 + backtest_df['Strategy_Return']).cumprod().iloc[-1]

print(f"Buy and Hold Growth Factor: {buy_hold_total:.2f}x")
print(f"AI Strategy Growth Factor: {strategy_total:.2f}x")

Buy and Hold Growth Factor: 1.20x
AI Strategy Growth Factor: 0.99x


In [ ]:
#Check how many times the model predicted UP (1) vs DOWN (0)
print(pd.Series(test_predictions).value_counts())


0    50
1    13
Name: count, dtype: int64


In [ ]:
import pandas as pd

#Extract probability scores for the UP class (Class 1)
# probabilities[:, 1] holds the probability (0.0 to 0.1) of Bitcoin going UP
probabilities = model.predict_proba(x_test)[:, 1]

#Define custom threshold (55% confidence required)
threshold = 0.55
custom_predictions = (probabilities > threshold).astype(int)

#Run backtest with custom threshold signals
backtest_df = pd.DataFrame(index=x_test.index)
backtest_df['Prediction'] = custom_predictions
backtest_df['Actual_Return'] = df.loc[x_test.index, 'Pct_change'].shift(-1)
backtest_df['Strategy_Return'] = backtest_df['Prediction'] * backtest_df['Actual_Return']

backtest_df = backtest_df.dropna()
buy_hold_total = (1 + backtest_df['Actual_Return']).cumprod().iloc[-1]
strategy_total = (1 + backtest_df['Strategy_Return']).cumprod().iloc[-1]

print("Signal breakdown with 55% threshold:")
print(pd.Series(custom_predictions).value_counts())
print(f"\nAI Strategy Growth Factor: {strategy_total:.2f}x")

Signal breakdown with 55% threshold:
0    55
1     8
Name: count, dtype: int64

AI Strategy Growth Factor: 1.03x


In [ ]:
from sklearn.ensemble import RandomForestClassifier

#Train model with hyperparameter constraints
model = RandomForestClassifier(n_estimators=100, max_depth=4, #limits depth to prevent overfitting
                               min_samples_leaf=5, #Requires atleast 5 samples per leaf
                               random_state=42)
model.fit(x_train, y_train)

#Check updated prediction distribution
test_predictions = model.predict(x_test)
print("Updated Predicition Distribution:")
print(pd.Series(test_predictions).value_counts())


Updated Predicition Distribution:
0    41
1    22
Name: count, dtype: int64


In [ ]:
#Align predictions with actual price returns
backtest_df = pd.DataFrame(index=x_test.index)
backtest_df['Prediction'] = test_predictions
backtest_df['Actual_Return'] = df.loc[x_test.index, 'Pct_change'].shift(-1)

#Strategy return (1 = hold Bitcoin return, 0 = stay in cash 0%)
backtest_df['Strategy_Return'] = backtest_df['Prediction'] * backtest_df['Actual_Return']

#calculate updated strategy growth factor
backtest_df = backtest_df.dropna()
buy_hold_total = (1 + backtest_df['Actual_Return']).cumprod().iloc[-1]
strategy_total = (1 + backtest_df['Strategy_Return']).cumprod().iloc[-1]

print(f"Buy and Hold Growth Factor: {buy_hold_total:.2f}x")
print(f"New AI Strategy Growth Factor: {strategy_total:.2f}x")



Buy and Hold Growth Factor: 1.20x
New AI Strategy Growth Factor: 1.11x


In [ ]:
import pandas as pd

#Map predictions: 1-> +1 (Long), 0 -> -1 (Short)
backtest_df = pd.DataFrame(index=x_test.index)
backtest_df['Position'] = pd.Series(test_predictions, index=x_test.index).map({1: 1, 0: -1})

#Align with actual returns
backtest_df['Actual_Return'] = df.loc[x_test.index, 'Pct_change'].shift(-1)
backtest_df['Strategy_Return'] = backtest_df['Position'] * backtest_df['Actual_Return']

#Calculate growth factors
backtest_df = backtest_df.dropna()
buy_hold_total = (1 + backtest_df['Actual_Return']).cumprod().iloc[-1]
strategy_total = (1 + backtest_df['Strategy_Return']).cumprod().iloc[-1]

print(f"Buy and Hold Growth Factor: {buy_hold_total:.2f}x")
print(f"Constrained AI Shorting Growth Factor: {strategy_total:.2f}x")

Buy and Hold Growth Factor: 1.20x
Constrained AI Shorting Growth Factor: 1.02x


In [ ]:
import pandas as pd
from sklearn.ensemble import RandomForestClassifier

#Compute 12-day and 26-day Exponential Moving Averages
ema_12 = df['Close'].ewm(span=12, adjust=False).mean()
ema_26 = df['Close'].ewm(span=26, adjust=False).mean()

#Calculate the MACD and Signal line
df['MACD'] = ema_12 - ema_26
df['MACD_Signal'] = df['MACD'].ewm(span=9, adjust=False).mean()

#Update feature set x with new indicators
features = ['Open', 'High', 'Low', 'Close', 'Volume', 'SMA_10', 'SMA_50', 'Pct_change', 'RSI', 'MACD', 'MACD_Signal']
x = df[features].dropna()
y = df.loc[x.index, 'Target']

#Re-split data (preserving time sequence)
split_idx = int(len(x) * 0.8)
x_train, x_test = x.iloc[:split_idx], x.iloc[split_idx:]
y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]

#Train constrained Random Forest on enriched features
model = RandomForestClassifier(n_estimators=100, max_depth=4, min_samples_leaf=5, random_state=42)
model.fit(x_train, y_train)

#Evaluate predictions
new_preds = model.predict(x_test)
print("Updated Prediction Distribution with MACD:")
print(pd.Series(new_preds).value_counts())

Updated Prediction Distribution with MACD:
0    51
1    12
Name: count, dtype: int64


In [ ]:
from numpy import short
import pandas as pd

#prepare backtest DataFrame
backtest_df = pd.DataFrame(index=x_test.index)
backtest_df['Prediction'] = new_preds
backtest_df['Actual_Return'] = df.loc[x_test.index, 'Pct_change'].shift(-1)

#Strategy 1: Long/Cash Mode (1 = Long, 0 = Cash)
backtest_df['Cash_Return'] = backtest_df['Prediction'] * backtest_df['Actual_Return']

#Strategy 2: Long/Short Mode (1 = Long, -1 = Short)
backtest_df['Position_Short'] = backtest_df['Prediction'].map({1: -1, 0: 1})
backtest_df['Short_Return'] = backtest_df['Position_Short'] * backtest_df['Actual_Return']

#Compute cummulative performance
backtest_df = backtest_df.dropna()
buy_hold_total = (1 + backtest_df['Actual_Return']).cumprod().iloc[-1]
cash_strategy_total = (1 + backtest_df['Cash_Return']).cumprod().iloc[-1]
short_Strategy_total = (1 + backtest_df['Short_Return']).cumprod().iloc[-1]

print(f"Buy and Hold Growth Factor: {buy_hold_total:.2f}x")
print(f"MACD Strategy (Cash Mode): {cash_strategy_total:.2f}x")
print(f"MACD Strategy (Short Mode): {short_Strategy_total:.2f}x")

Buy and Hold Growth Factor: 1.20x
MACD Strategy (Cash Mode): 1.08x
MACD Strategy (Short Mode): 1.01x


In [ ]:
import pandas as pd

#Pair feature names with their calculated importance scores
importance_df = pd.DataFrame({
    'Feature': features,
    'Importance': model.feature_importances_
}).sort_values('Importance', ascending=False)

#Print the sorted table
print("Feature Importance Ranking:")
print(importance_df.to_string(index=False))

Feature Importance Ranking:
    Feature  Importance
 Pct_change    0.133811
        RSI    0.121295
MACD_Signal    0.101755
      Close    0.095095
        Low    0.089545
     SMA_50    0.085126
       Open    0.083519
       High    0.082869
     Volume    0.081886
       MACD    0.065210
     SMA_10    0.059891


In [ ]:
import numpy as np
from sklearn.model_selection import TimeSeriesSplit

# Initialize TimeSeriesSplit with 4 folds
tscv = TimeSeriesSplit(n_splits=4)
fold_returns = []

print("Walk-Forward Validation Results (Cash Mode):")
for fold, (train_idx, test_idx) in enumerate(tscv.split(x)):
  x_tr, x_te = x.iloc[train_idx], x.iloc[test_idx]
  y_tr, y_te = y.iloc[train_idx], y.iloc[test_idx]

  # Retrain model on fold historical window
  fold_model = RandomForestClassifier(n_estimators=100, max_depth=4, min_samples_leaf=5, random_state=42)
  fold_model.fit(x_tr, y_tr)
  preds = fold_model.predict(x_te)

  # Calculate the fold performance
  fold_df = pd.DataFrame(index=x_te.index)
  fold_df['Prediction'] = preds
  fold_df['Actual_Return'] = df.loc[x_te.index, 'Pct_change'].shift(-1)
  fold_df = fold_df.dropna()

  fold_growth = (1 + (fold_df['Prediction'] * fold_df['Actual_Return'])).cumprod().iloc[-1]
  fold_returns.append(fold_growth)
  print(f"Fold {fold + 1} Growth Factor: {fold_growth:.2f}x")

print(f"\nAverage Growth Factor: {np.mean(fold_returns):.2f}x")

Walk-Forward Validation Results (Cash Mode):
Fold 1 Growth Factor: 1.01x
Fold 2 Growth Factor: 0.99x
Fold 3 Growth Factor: 1.01x
Fold 4 Growth Factor: 1.08x

Average Growth Factor: 1.02x


In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import TimeSeriesSplit

# Initialize TimeSeriesSplit with 4 folds
tscv = TimeSeriesSplit(n_splits=4)
fold_returns = []
stop_loss_limit = -0.02 # Cap daily trade losses at -2%

print("Walk-Forward Validation with -2% Stop-Loss (Cash Mode):")
for fold, (train_idx, test_idx) in enumerate(tscv.split(x)):
  x_tr, x_te = x.iloc[train_idx], x.iloc[test_idx]
  y_tr, y_te = y.iloc[train_idx], y.iloc[test_idx]

   # Retrain model on fold historical window
  fold_model = RandomForestClassifier(n_estimators=100, max_depth=4, min_samples_leaf=5, random_state=42)
  fold_model.fit(x_tr, y_tr)
  preds = fold_model.predict(x_te)

  # Calculate the fold performance
  fold_df = pd.DataFrame(index=x_te.index)
  fold_df['Prediction'] = preds
  fold_df['Actual_Return'] = df.loc[x_te.index, 'Pct_change'].shift(-1)
  fold_df = fold_df.dropna()

  # Calculate the raw return
  raw_return = fold_df['Prediction'] * fold_df['Actual_Return']

  # Apply stop-loss filter: cap any daily loss worse than 2% at -0.02
  fold_df['Protected_Return'] = raw_return.apply(lambda r: max(r, stop_loss_limit))

  # Compound cummulative growth factor
  fold_growth = (1 + fold_df['Protected_Return']).cumprod().iloc[-1]
  fold_returns.append(fold_growth)
  print(f"Fold {fold + 1} Growth Factor: {fold_growth:.2f}x")

print(f"\nAverage Growth Factor: {np.mean(fold_returns):.2f}x")

Walk-Forward Validation with -2% Stop-Loss (Cash Mode):
Fold 1 Growth Factor: 1.02x
Fold 2 Growth Factor: 1.03x
Fold 3 Growth Factor: 1.04x
Fold 4 Growth Factor: 1.10x

Average Growth Factor: 1.05x


In [ ]:
import yfinance as yf
import pandas as pd

# Fetch recent Bitcoin data (last 100 days to allow for moving averages)
btc_live = yf.download('BTC-USD', period='100d', interval='1d')

if isinstance(btc_live.columns, pd.MultiIndex):
  btc_live.columns = btc_live.columns.get_level_values(0)

# Compute technical indicators matching training setup
btc_live['Pct_change'] = btc_live['Close'].pct_change()
btc_live['SMA_10'] = btc_live['Close'].rolling(window=10).mean()
btc_live['SMA_50'] = btc_live['Close'].rolling(window=50).mean()

# RSI
delta = btc_live['Close'].diff()
gain = (delta.where(delta > 0, 0)).rolling(window=14).mean()
loss = (-delta.where(delta < 0, 0)).rolling(window=14).mean()
btc_live['RSI'] = 100 - (100 / (1 + (gain / loss)))

# MACD
ema_12 = btc_live['Close'].ewm(span=12, adjust=False).mean()
ema_26 = btc_live['Close'].ewm(span=26, adjust=False).mean()
btc_live['MACD'] = ema_12 - ema_26
btc_live['MACD_Signal'] = btc_live['MACD'].ewm(span=9, adjust=False).mean()

# Extract the latest complete feature row
latest_features = btc_live[features].iloc[[-1]]

# Generate live model prediction
live_prediction = model.predict(latest_features)[0]
signal = "LONG (Hold BTC)" if live_prediction == 1 else "CASH (Stay in cash)"

print(f"Latest Market Date: {latest_features.index[0].strftime('%Y-%m-%d')}")
print(f"Latest Close Price: ${btc_live['Close'].iloc[-1]:,.2f}")
print(f"Live AI Signal: {signal}")

/tmp/ipykernel_675/1581903205.py:5: FutureWarning: YF.download() has changed argument auto_adjust default to True
  btc_live = yf.download('BTC-USD', period='100d', interval='1d')
[*********************100%***********************]  1 of 1 completed

Latest Market Date: 2026-10-22
Latest Close Price: $84,166.61
Live AI Signal: LONG (Hold BTC)


In [ ]:
import os
import pandas as pd

# prepare today's log entry
log_entry = pd.DataFrame(
    [{
        'Timestamp': latest_features.index[0].strftime('%Y-%m-%d'),
        'Close_Price': round(btc_live['Close'].iloc[-1], 2),
        'Prediction': live_prediction,
        'Signal': "LONG" if live_prediction == 1 else "CASH"
    }]
)

# Append to CSV file (creates file if it doesnt exist)
log_file = 'trading_signals_log.csv'
if not os.path.exists(log_file):
    log_entry.to_csv(log_file, index=False)
else:
    log_entry.to_csv(log_file, mode='a', header=False, index=False)

# Display updated trading log
print("Current Trading Log:")
print(pd.read_csv(log_file))

Current Trading Log:
    Timestamp  Close_Price  Prediction Signal
0  2026-10-22     84166.61           1   LONG


**Algorithmic Bitcoin Trading Bot with ML & Risk Management**

An end-to-end Machine learning pipeline that predicts Bitcoin price momentum using technical indicators, evaluates stability through Walk-Forward cross-validation, and applies risk management filters for automated signal generation.

**System Architecture**


1.   Feature Engineering: Standard technical indicators ('SMA', 'RSI', 'Pct_change', 'MACD')
2.   Constrained Model: Regularized 'RandomForestClassifier' ('max_depth=4', 'min_samples_leaf=5') to prevent overfitting.


1.   Validation: 4-fold 'TimeSeriesSplit' across sequential market regimes.
2.   Risk Controls: -2% daily stop_loss execution filter.


1.   Live Pipeline & Logging: Real-time market data ingestionvia 'yfinace' with CSV logging.